# 04: Experimental Evaluation (Locked Test Window)

This notebook evaluates the models on the untouched test set. We calculate Recall@K, NDCG@K, Hit Rate, and Coverage.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.db import query_to_dataframe
from src.recommenders import PopularityRecommender, ItemCFRecommender, UserCFRecommender, AssociationRuleRecommender, HybridRecommender
from src.evaluation import evaluate_model, stratify_customers_by_history

## 1. Load Locked Test Data

In [ ]:
query = """
SELECT i.customer_id, ii.stock_code, i.invoice_date
FROM invoices i
JOIN invoice_items ii ON i.invoice_no = ii.invoice_no
WHERE i.is_cancellation = 0
"""
df = query_to_dataframe(query)
df["invoice_date"] = pd.to_datetime(df["invoice_date"])

train_end = pd.to_datetime("2011-10-31") # Validation was up to Oct 31

train_and_val_df = df[df["invoice_date"] < train_end]
test_df = df[df["invoice_date"] >= train_end]
print("Training + Val history shape:", train_and_val_df.shape)
print("Test (Ground Truth) shape:", test_df.shape)

## 2. Fit Models on Train+Val, Evaluate on Test

In [ ]:
pop_rec = PopularityRecommender()
pop_rec.fit(train_and_val_df)

print("Evaluating Popularity Baseline...")
pop_metrics = evaluate_model(pop_rec, train_and_val_df, test_df, k=10)
print("Popularity Metrics:", pop_metrics)

## 3. History Stratification (Cold-Start Experiment)

In [ ]:
strata = stratify_customers_by_history(train_and_val_df)
print("Strata counts:", {k: len(v) for k, v in strata.items()})

# Evaluate specifically on the low_history cohort
low_history_test = test_df[test_df["customer_id"].isin(strata["low_history"])]
print("Evaluating Popularity on Low History cohort...")
pop_low_metrics = evaluate_model(pop_rec, train_and_val_df, low_history_test, k=10)
print("Low History Popularity Metrics:", pop_low_metrics)